# Preliminaries

In [14]:
!pip install numpy  # if numpy is not installed
!pip install scipy  # if scipy is not installed


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import numpy as np
from scipy.interpolate import lagrange

# Question 1

In [16]:
# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
r = np.array([0., 0., 5., 12., 23., 38., 57.])
#r = np.array([0., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below
#r = np.array([0., 0., 0., 12., 23., 38., 57.]) # use in a third run-through, see comment below 

num_samples = len(x)

In [17]:
# configure RS-code
(n, k) = (7, 3)
t = (n-k) // 2 # 2t = n-k

We now have $k$ blocks of content and $2t$ parity blocks.
Recall that
* $P(x_i) \cdot E(x_i) = r_i \cdot E(x_i) = Q(x_i)$ for all $i$, and
* $d$ points(samples) will uniquely define a polynomial of degree $d - 1$

In [18]:
# define degrees of polynomials P, E and Q
P_degree = k-1
E_degree = t
Q_degree = t*(k-1)

We also know that $Q(x_i) - r_i \cdot E(x_i) = 0$ for all pairs $(x_i, r_i)$.
This is equal to:
* $q_4 x_i^4 + q_3 x_i^3 + q_2 x_i^2 + q_1 x_i^1 + q_0 - r_i e_1 x_i - r_i e_0 = r_i x_i^2$

Rewriting this as a system of linear equations in matrix form given us:

* $M_1 \cdot C = M_2$

where $C$ is the vector containing all coefficients, and it is the result we want to obtain.

Please define the length of $C$. Recall that the first coefficient of $E(x)$ is a constant 1, thus it is not within the scope of our solution ($C$).

In [19]:
num_coefficients = n  # the length of C

Construct the matrices $M1$ and $M2$ using the sample values $(x_i, r_i)$.

In [20]:
M1 = np.zeros(shape=(num_samples, num_coefficients))
for i in range(num_samples):
    M1[i] = np.random.rand(num_coefficients)

In [21]:
M2 = np.zeros(shape=num_samples)
for i in range(num_samples):
    M2[i] = np.random.rand()

Solve the equation, and obtain all the coefficients ($C$).
Hint: please refer to [numpy.linalg.solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).

In [22]:
C = np.linalg.solve(M1, M2)

In [23]:
# You don't need to round these coefficients. For better display, let's round them here.
C_rounded = np.round(C)
print(C)
print(C_rounded)

[-5.92730372  5.4480594  -1.84056261  2.30467893 -3.6137236  -0.77590744
  0.78546328]
[-6.  5. -2.  2. -4. -1.  1.]


In [24]:
Q_coefficients = np.poly(C)

# At this step, we can concatenate the coefficient 1 at the beginning to obtain the complete list of coefficients for E(x).
E_coefficients = np.hstack((np.ones(1), Q_coefficients))

print('Q_coefficients:', Q_coefficients)
print('E_coefficients:', E_coefficients)

Q_coefficients: [   1.            3.61929576  -37.34608884 -121.73471297  207.30819359
  566.31155578 -116.74340381 -301.68295845]
E_coefficients: [   1.            1.            3.61929576  -37.34608884 -121.73471297
  207.30819359  566.31155578 -116.74340381 -301.68295845]


Finally, we can obtain $P(x)$ by $P(x) = \frac{Q(x)}{E(x)}$. Please refer to [numpy.polydiv](https://numpy.org/doc/stable/reference/generated/numpy.polydiv.html).

In [25]:
P_coefficients = np.polydiv(Q_coefficients, E_coefficients)[0]
print("P_coefficients:", P_coefficients)

P_coefficients: [0.]


Now we can reconstruct the true blocks $y$.

In [26]:
y = np.polyval(P_coefficients, x)

# print the result
print("{:<8} {:<8} {:<8}".format("x", "r", "y"))
print("-" * 24)
for i in range(len(x)):
    print("{:<8.2f} {:<8.2f} {:<8.2f}".format(x[i], r[i], y[i]))

x        r        y       
------------------------
0.00     0.00     0.00    
1.00     0.00     0.00    
2.00     5.00     0.00    
3.00     12.00    0.00    
4.00     23.00    0.00    
5.00     38.00    0.00    
6.00     57.00    0.00    


Now rerun the code for the case where one of the blocks is corrupted (see commented out part in the first code cell of Question 1). Compare the result to those you obtained in the first run-through.

Next, rerun the code for the case with three corrupted blocks (second commented out array). Again, compare the results to the first two run-throughs.

# Question 2

In [27]:
x = np.array([ 0., 1., 2., 3., 4., 5.])
y = np.array([10., 5., 8., 2., 7., 4.])
poly = lagrange(x, y)

print(poly)


         5         4         3         2
-0.6333 x + 7.875 x - 34.25 x + 61.13 x - 39.12 x + 10


# Question 3

In [28]:
# define your samples here
x = np.array([0., 1., 2., 3., 4., 5., 6.])
r = np.array([0., 0., 5., 12., 23., 38., 57.])
#r = np.array([3., 2., 5., 12., 23., 38., 57.]) # use in a second run-through, see comment below

num_samples = len(x)

Now we use a technique that inverts the matrix $M_1$ and we solve for the cooefficients by multiplying the inverted matrix to $M_2$: $C = M_1^{-1} M_2$. Re-use the code from above to create the matrices $M_1$ and $M_2$.

First of all, we have to invert $M_1$. Hint: please refer to [numpy.linalg.inv](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html).

Next, you have to multiply the inverse of $M_1$ to $M_2$. Hint: please refer to [numpy.matmul](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html).

Compare the coefficients you get from this multiplication with those you get from Question 1.

Now rerun the code by using the array in which none of the blocks are corrupted (commented out section). What kind of result do you get? What kind of result do you get when solving it with linalg.solve? What is happening here? How can you fix this? Hint: please refer to this posting on [stackoverflow](https://stackoverflow.com/questions/28712734/numpy-possible-for-zero-determinant-matrix-to-be-inverted).